gk/ball Position Extraction csv

In [7]:
import json
import pandas as pd
import os

# Path to your dataset folder
dataset_path = "../datasets/football-players-detection.v1i.coco"
output_path = "../output"

# Folders for train, valid, test
splits = ['train', 'valid', 'test']

# Function to extract positions from COCO JSON for each folder
def extract_positions_from_coco(json_file):
    with open(json_file, "r") as f:
        data = json.load(f)

    # Prepare a list to store time-series data
    time_series_data = []

    # Loop through each image (frame)
    for image in data["images"]:
        frame_id = image["id"]
        filename = image["file_name"]

        # Find corresponding annotations for each image (frame)
        annotations = [ann for ann in data["annotations"] if ann["image_id"] == frame_id]

        # Initialize positions for goalkeeper and ball
        gk_x, gk_y, ball_x, ball_y = None, None, None, None

        # Loop through annotations and extract positions
        for ann in annotations:
            category_id = ann["category_id"]  # Category ID (goalkeeper or ball)
            bbox = ann["bbox"]  # Bounding box: [x, y, width, height]
            obj_x = bbox[0] + bbox[2] / 2  # X center of the bounding box
            obj_y = bbox[1] + bbox[3] / 2  # Y center of the bounding box

            # Identify the object based on category ID
            if category_id == 2:  # Goalkeeper (ID 2)
                gk_x, gk_y = obj_x, obj_y
            elif category_id == 1:  # Ball (ID 1)
                ball_x, ball_y = obj_x, obj_y

        # Store the data in the time_series_data list
        time_series_data.append([frame_id, filename, gk_x, gk_y, ball_x, ball_y])

    # Convert to a DataFrame
    df = pd.DataFrame(time_series_data, columns=["Frame", "Filename", "GK_x", "GK_y", "Ball_x", "Ball_y"])

    return df

# Process each split (train, valid, test)
for split in splits:
    # Path to the current split folder
    split_path = os.path.join(dataset_path, split)
    
    # Path to the annotations JSON file for the current split
    json_file = os.path.join(split_path, "_annotations.coco.json")

    if os.path.exists(json_file):
        # Extract positional data from the current split's JSON file
        df = extract_positions_from_coco(json_file)

        # Ensure output folder exists for this split
        output_split_path = os.path.join(output_path, split)
        os.makedirs(output_split_path, exist_ok=True)

        # Save the time-series data as CSV
        output_csv_file = os.path.join(output_split_path, f"{split}_time_series.csv")
        df.to_csv(output_csv_file, index=False)

        print(f"Positional data for {split} split saved as {output_csv_file}!")

    else:
        print(f"Annotations JSON file not found for {split} split.")

print("Processing completed for all splits.")


Positional data for train split saved as ../output\train\train_time_series.csv!
Positional data for valid split saved as ../output\valid\valid_time_series.csv!
Positional data for test split saved as ../output\test\test_time_series.csv!
Processing completed for all splits.
